# PMALA on a real patient fiber field

Interactive test of the Laplace-preconditioned MALA sampler (`PMALA`) for patient 01 (`example_data/raw/`). The posterior is the one of `patient_map.ipynb` and `patient_laplace.ipynb`, and the chain is preconditioned by the low-rank Laplace approximation $\mathcal N(m_\mathrm{MAP}, \Gamma)$ at its MAP estimate (`map_patient.npy`, next to this notebook). The proposal is the Langevin step with drift $-\frac{2\delta}{2+\delta}\,\Gamma\,(\nabla\Phi + \Gamma_0^{-1}(u - \bar m))$ and noise $\mathcal N(0, \Gamma)$; the prior mean $\bar m$ is moved into the potential, so it need not be zero.

The data and the prior mean are regenerated exactly as in `patient_map.ipynb` (same observation settings, seeds and prior mean constant), and the loss is checked against the MAP estimate below. Change the settings in all three notebooks together.

Every chain step needs a forward and an adjoint solve, so chains are short here. `RUN_SAMPLER = False` reads the stored chain instead. Posterior is far from Gaussian: expect step widths around $0.05$ or smaller (`STEP_WIDTH`), and check the acceptance rate before reading anything into the statistics.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
import zarr
from ls_bayesian.common.logging import BaseLogger, LoggerSettings
from ls_bayesian.lowrank.eigensolvers import RandomizedDoublePassSolver
from ls_bayesian.mcmc.sampler import SamplerSettings

from bayes_cep.laplace.builder import LaplaceBuilder, LaplaceSettings
from bayes_cep.mcmc.builder import (
    SAMPLE_STORE_NAME,
    MCMCAlgorithmName,
    MCMCBuilder,
    MCMCSettings,
    ZarrStorageSettings,
)
from bayes_cep.mcmc.measures import LowRankGaussianMeasure
from bayes_cep.mesh.interpolation import NearestNeighborInterpolationStrategy
from bayes_cep.mesh.io import load_pyvista_mesh
from bayes_cep.posterior.builder import PosteriorBuilder, PosteriorSettings
from bayes_cep.posterior.eikonal_map import EikonalParameterToSolutionMap
from bayes_cep.posterior.likelihood import LikelihoodSettings
from bayes_cep.posterior.prior import PriorParameters
from bayes_cep.preprocessing.angle_field import build_angle_field_from_fiber_field
from bayes_cep.preprocessing.synthetic_observations import (
    ObservationSamplingSettings,
    generate_synthetic_observations,
)
from bayes_cep.statistics.axial_diagnostics import (
    compute_axial_autocorrelation,
    compute_axial_effective_sample_size,
)
from bayes_cep.statistics.axial_error_metrics import compute_error_by_distance
from bayes_cep.statistics.axial_statistics import (
    compute_axial_data_diff,
    compute_axial_mean_and_variance,
    compute_standard_deviation_degrees,
    wrap_axial_angles,
)
from single_runs.reference import (
    NOISE_VARIANCE,
    NUM_OBSERVATIONS,
    OBSERVATION_SEED,
    reference_eikonal_settings,
)

pv.set_jupyter_backend("trame")

## Settings

Prior, observations and prior mean as in `patient_map.ipynb`. `NUM_EIGENVALUES` is the rank of the Laplace approximation (at most the number of observations); directions beyond the smallest kept eigenvalue keep their wide prior variance, which makes the preconditioner too bold there and the sampler reject. `STEP_WIDTH` is the $\delta$ of PMALA.

In [ ]:
PATIENT_DIR = Path.cwd().parent / "example_data" / "raw"
MAP_PATH = Path.cwd() / "map_patient.npy"
CHAIN_DIR = Path.cwd().parent / "working_data" / "patient_pmala_notebook"  # gitignored

PRIOR = PriorParameters(kappa=0.5, tau=0.5, seed=0)
OBSERVATIONS = ObservationSamplingSettings(
    num_observations=NUM_OBSERVATIONS, noise_variance=NOISE_VARIANCE, seed=OBSERVATION_SEED
)
MEAN_CONSTANT = None  # as in patient_map.ipynb: None scans for the constant that fits best

NUM_EIGENVALUES = 1000
OVERSAMPLING = 20
MINIMUM_EIGENVALUE = 1e-6

STEP_WIDTH = 2e-4
NUM_SAMPLES = 1000
SEED = 0
RUN_SAMPLER = True

## Data and MAP estimate

The ground truth is the patient field; the activation times are synthetic (forward-solved, sparsely sampled, noisy), as in `patient_map.ipynb`.

In [ ]:
mesh = load_pyvista_mesh(PATIENT_DIR / "mesh.vtu")
basis_vectors = np.load(PATIENT_DIR / "basis_vecs.npy")
fiber_field = np.load(PATIENT_DIR / "fiber_field.npy")
eikonal_settings = reference_eikonal_settings()
interpolation_strategy = NearestNeighborInterpolationStrategy()
connectivity = mesh.cells.reshape(-1, 4)[:, 1:]

ground_truth = build_angle_field_from_fiber_field(mesh, fiber_field, basis_vectors)
forward_map = EikonalParameterToSolutionMap(
    mesh, basis_vectors, eikonal_settings, interpolation_strategy
)
observed_vertex_indices, observed_activation_times = generate_synthetic_observations(
    forward_map, ground_truth, OBSERVATIONS
)
if MEAN_CONSTANT is None:
    candidates = np.linspace(-np.pi / 2, np.pi / 2, 25, endpoint=False)
    misfits = [
        np.sum(
            (
                forward_map.evaluate_forward(np.full(mesh.n_points, constant))[observed_vertex_indices]
                - observed_activation_times
            )
            ** 2
        )
        for constant in candidates
    ]
    mean_constant = float(candidates[np.argmin(misfits)])
else:
    mean_constant = MEAN_CONSTANT
prior_mean = np.full(mesh.n_points, mean_constant)

map_estimate = np.load(MAP_PATH)
if map_estimate.shape != (mesh.n_points,):
    raise ValueError(f"MAP estimate has shape {map_estimate.shape}, expected ({mesh.n_points},)")
print(f"{mesh.n_points} vertices, {mesh.n_cells} triangles, {observed_vertex_indices.size} observed")
print(f"prior mean: constant {mean_constant:.2f} rad")

## Posterior and Laplace approximation

`build_components` returns the likelihood, forward map and prior uncomposed (the Hessian needs the first two); `build` gives the `LogPosterior` the sampler uses. The MAP estimate belongs to this posterior if its loss is far below that of the prior mean.

In [ ]:
posterior_builder = PosteriorBuilder(
    PosteriorSettings(
        mesh=mesh,
        basis_vectors=basis_vectors,
        prior_mean_vector=prior_mean,
        prior_parameters=PRIOR,
        eikonal_settings=eikonal_settings,
        likelihood_settings=LikelihoodSettings(
            num_vertices=mesh.n_points,
            observed_vertex_indices=observed_vertex_indices,
            data_vector=observed_activation_times,
            noise_variance=OBSERVATIONS.noise_variance,
        ),
    ),
    interpolation_strategy=interpolation_strategy,
)
components = posterior_builder.build_components()
log_posterior = posterior_builder.build()

for name, field in (("MAP estimate", map_estimate), ("prior mean", prior_mean), ("patient field", ground_truth)):
    likelihood_cost = log_posterior.evaluate_likelihood_cost(field)
    prior_cost = log_posterior.evaluate_prior_cost(field)
    print(f"{name:<14} loss {likelihood_cost + prior_cost:>12.1f}  (likelihood {likelihood_cost:>10.1f}, prior {prior_cost:>10.1f})")

laplace_settings = LaplaceSettings(
    eigensolver=RandomizedDoublePassSolver(oversampling=OVERSAMPLING, seed=0),
    num_eigenvalues=NUM_EIGENVALUES,
    minimum_eigenvalue=MINIMUM_EIGENVALUE,
    sample_seed=0,
)
approximation = LaplaceBuilder(laplace_settings, components).build(map_estimate)
eigenvalues = approximation.eigenpairs.eigenvalues
laplace_measure = LowRankGaussianMeasure(approximation.gaussian)

print(f"kept {eigenvalues.size} of {NUM_EIGENVALUES} eigenpairs")
print(f"eigenvalues from {eigenvalues[0]:.3e} down to {eigenvalues[-1]:.3e}")

## Build the sampler

`MCMCSettings.algorithm = PMALA` selects the preconditioned MALA; `MCMCBuilder` takes the Laplace measure as `approximation`. The chain is written to a Zarr store in `CHAIN_DIR`; it is overwritten when the sampler is built. `TRACKED_COMPONENTS` are logged with their running mean next to the running acceptance rate.

In [ ]:
TRACKED_COMPONENTS = (0, 42, 100, 1000)

if RUN_SAMPLER:
    CHAIN_DIR.mkdir(parents=True, exist_ok=True)
    mcmc_builder = MCMCBuilder(
        MCMCSettings(
            algorithm=MCMCAlgorithmName.PMALA,
            step_width=STEP_WIDTH,
            tracked_components=TRACKED_COMPONENTS,
            storage=ZarrStorageSettings(chunk_size=100),
        ),
        log_posterior,
        output_dir=CHAIN_DIR,
        approximation=laplace_measure,
    )

## Run the chain

The chain starts at the MAP estimate. A chain that does not move has an acceptance rate of about $1/N$ (the initial state counts as accepted): lower `STEP_WIDTH` until it moves, then raise it.

In [ ]:
if RUN_SAMPLER:
    with BaseLogger(LoggerSettings(), prefix="pmala") as logger:
        sampler = mcmc_builder.build(logger=logger)
        sampler.run(
            initial_state=map_estimate,
            settings=SamplerSettings(num_samples=NUM_SAMPLES, log_interval=1),
            seed=SEED,
        )
    print(f"final acceptance rate {float(mcmc_builder.outputs[0].all_values[-1]):.3f}")

## Acceptance rate and potential along the chain

The running acceptance rate, and the posterior loss of the chain states at every `LOSS_STRIDE`-th sample (a likelihood and a prior evaluation each, i.e. one forward solve). The chain starts at the MAP estimate, the minimum of the loss; a chain that stays near it is not exploring, one that drifts far above the typical posterior loss has left the mode.

In [ ]:
LOSS_STRIDE = 20

chain = zarr.open_group(CHAIN_DIR / SAMPLE_STORE_NAME, mode="r")["data"]
num_chain_samples = chain.shape[0]
print(f"chain: {num_chain_samples} samples of {chain.shape[1]} vertices")

loss_indices = np.arange(0, num_chain_samples, LOSS_STRIDE)
losses = np.array(
    [
        log_posterior.evaluate_likelihood_cost(state) + log_posterior.evaluate_prior_cost(state)
        for state in np.asarray(chain.get_orthogonal_selection((loss_indices, slice(None))))
    ]
)

fig, (rate_axis, loss_axis) = plt.subplots(1, 2, figsize=(10, 3.5))
if RUN_SAMPLER:
    rate_axis.plot(np.asarray(mcmc_builder.outputs[0].all_values))
rate_axis.set_xlabel("iteration")
rate_axis.set_ylabel("running acceptance rate")
loss_axis.plot(loss_indices, losses, marker=".")
loss_axis.set_xlabel("iteration")
loss_axis.set_ylabel("posterior loss")
fig.tight_layout()

## Traces

The angle at the tracked vertices along the chain. A flat trace has not moved from the MAP estimate.

In [ ]:
TRACED_VERTICES = TRACKED_COMPONENTS

fig, axes = plt.subplots(len(TRACED_VERTICES), 1, figsize=(8, 2 * len(TRACED_VERTICES)), sharex=True)
for axis, vertex in zip(axes, TRACED_VERTICES, strict=True):
    axis.plot(np.asarray(chain[:, vertex]))
    axis.axhline(map_estimate[vertex], color="gray", linewidth=0.8)
    axis.set_ylabel(f"vertex {vertex} [rad]")
axes[-1].set_xlabel("iteration")
fig.tight_layout()

## Autocorrelation and effective sample size

Axial autocorrelation (on $\sin(2(\theta - \mu))$) of the traced vertices after the burn-in, and the effective sample size over a random subset of vertices (all of them are possible but slow here). A slowly decaying function means strongly correlated samples.

In [ ]:
BURN_IN = 100
MAX_LAG = 300
NUM_VERTICES_FOR_ESS = 200

traces = np.asarray(chain.get_orthogonal_selection((slice(None), list(TRACED_VERTICES))))
autocorrelation = compute_axial_autocorrelation(traces, MAX_LAG, BURN_IN)

fig, axis = plt.subplots(figsize=(6, 4))
axis.axhline(0.0, color="gray", linewidth=0.8)
for vertex, function in zip(TRACED_VERTICES, autocorrelation.T, strict=True):
    axis.plot(function, label=f"vertex {vertex}")
axis.set_xlabel("lag")
axis.set_ylabel("autocorrelation")
axis.legend()
fig.tight_layout()

vertex_subset = np.sort(
    np.random.default_rng(1).choice(mesh.n_points, NUM_VERTICES_FOR_ESS, replace=False)
)
subset = np.asarray(chain.get_orthogonal_selection((slice(None), vertex_subset)))
effective_sample_size = compute_axial_effective_sample_size(subset, MAX_LAG, BURN_IN)
print(f"effective sample size: min {np.nanmin(effective_sample_size.effective_sample_size):.1f}, "
      f"median {np.nanmedian(effective_sample_size.effective_sample_size):.1f}")

## Posterior mean and standard deviation

The axial mean and variance of the chain after the burn-in; the variance is shown as a standard deviation in degrees. The mean is plotted per simplex and wrapped, as the forward model sees the field (interpolating the angle over a cell would run through the whole colormap at the seam $\pm\pi/2$). The MAP estimate and the patient field are shown for comparison.

In [ ]:
chain_mean, chain_variance = compute_axial_mean_and_variance(chain, burn_in=BURN_IN)
chain_std = compute_standard_deviation_degrees(chain_variance)
print(f"axial std: mean {chain_std.mean():.2f}, max {chain_std.max():.2f} deg")

vertex_to_simplex_matrix = interpolation_strategy.assemble_matrix(mesh.points, connectivity)
clim = (-np.pi / 2, np.pi / 2)
for name, field in (
    ("ground truth (patient)", ground_truth),
    ("MAP estimate", map_estimate),
    ("PMALA posterior mean", chain_mean),
):
    plot_mesh = mesh.copy()
    plot_mesh.cell_data["angle [rad]"] = wrap_axial_angles(vertex_to_simplex_matrix @ field)
    plotter = pv.Plotter()
    plotter.add_mesh(plot_mesh, scalars="angle [rad]", cmap="hsv", clim=clim)
    plotter.add_text(name)
    plotter.show()

plot_mesh = mesh.copy()
plot_mesh.point_data["axial std [deg]"] = chain_std
plotter = pv.Plotter()
plotter.add_mesh(plot_mesh, scalars="axial std [deg]", cmap="viridis")
plotter.add_text("posterior axial standard deviation [deg]")
plotter.show()

## Error against the patient field

The axial error of the posterior mean and of the MAP estimate relative to the patient field, in degrees, and the standardized error $z = (\bar m_\mathrm{chain} - m_\mathrm{truth})/\sigma$ with the chain standard deviation (fractions near 0.68 / 0.95 within 1 / 2 sigma are calibrated). The standard deviation is only meaningful once the chain has mixed.

In [ ]:
chain_error = compute_axial_data_diff(chain_mean, ground_truth)
map_error = compute_axial_data_diff(map_estimate, ground_truth)
print(f"mean absolute axial error: chain mean {np.degrees(np.abs(chain_error).mean()):.2f} deg, "
      f"MAP {np.degrees(np.abs(map_error).mean()):.2f} deg")

standardized_error = chain_error / np.radians(np.maximum(chain_std, 1e-6))
for num_sigma, nominal in ((1, 0.683), (2, 0.954)):
    print(f"within {num_sigma} sigma: {np.mean(np.abs(standardized_error) <= num_sigma):.3f} "
          f"(Gaussian: {nominal:.3f})")

fig, axis = plt.subplots(figsize=(6, 4))
bins = np.linspace(0, 90, 46)
axis.hist(np.degrees(np.abs(map_error)), bins=bins, alpha=0.6, label="MAP")
axis.hist(np.degrees(np.abs(chain_error)), bins=bins, alpha=0.6, label="chain mean")
axis.set_xlabel("absolute axial error [deg]")
axis.legend()
fig.tight_layout()